# Agentic RAG

In [1]:
from dotenv import load_dotenv
from openai import OpenAI

In [2]:
load_dotenv()

True

## Retrieval-Augmented Generation
LLM models are trained on large amounts of publicly available datasets. The information found in these datasets constitutes the intrinsic knowledge that a model can reason over and answer questions about. To generate business value, organizations need AI models to interact with proprietary data which was not available during model training.

RAG is a method for giving an LLM access to an external knowledge base that extends to model's capabilities to fit a specific purpose. In a RAG flow, the user query is first used to search for potential matches in the external database (retrieval). The matching documents form the context which the LLM will use to provide more specific and accurate answers than it would without the additional knowledge (augmentation).

In [3]:
def get_response(client, prompt):
	response = client.chat.completions.create(
		model="gpt-4o-mini",
		messages=prompt,
		user="llm-zoomcamp",
		stream=False
	)

	return response

In [4]:
openai_client = OpenAI()

In [7]:
prompt = [{ 'role': 'user', 'content': "Who won yesterday's NBA finals game?" }]
get_response(openai_client, prompt).choices[0].message.content

"I'm sorry, but I don't have access to real-time data, including sports scores or updates. You can check the latest NBA results on sports news websites or apps for the most current information."

## Dataset

In [8]:
import requests

In [9]:
docs_url = 'https://datatalks.club/faq/json/courses.json'
response = requests.get(docs_url)
courses_raw = response.json()

In [10]:
documents = []
url_prefix = 'https://datatalks.club/faq'

for course in courses_raw:
	course_url = f'{url_prefix}/{course["path"]}'

	course_response = requests.get(course_url)
	course_response.raise_for_status()
	course_data = course_response.json()

	documents.extend(course_data)

In [11]:
len(documents)

1342

In [12]:
documents[0]

{'id': '9e508f2212',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: When does the course start?',
 'answer': "A new cohort runs roughly January–April every year. For the current cohort's exact start date and registration link, check the [course repo README](https://github.com/DataTalksClub/data-engineering-zoomcamp).\n\n- Register via the link in the course repo before the cohort starts.\n- Join the [course Telegram channel](https://t.me/dezoomcamp) for announcements.\n- Join DataTalks.Club's [Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` channel."}

## Retrieval
Context scope limitation
- lowers LLM costs by reducing the amount of tokens passed to the model
- minimizes the probability of irrelevant model outputs
- enables faster model responses

Popular search engines:
- Lucine
- ElsticSearch
- Apache Solr
- minsearch

In [13]:
from minsearch import Index

In [14]:
def fit_index(text_fields, keyword_fields, documents):
	index  = Index(
		text_fields=text_fields,
		keyword_fields=keyword_fields
	)

	index.fit(documents)

	return index

In [15]:
index = fit_index(
	text_fields=['course', 'section', 'answer'],
	keyword_fields=['course'],
	documents=documents
)

In [16]:
def search(index, question):
	return index.search(
		question,
		filter_dict={'course': 'llm-zoomcamp'},
		num_results=5,
		boost_dict={'question': 2.0, 'section': .5}
	)

In [17]:
question = "i just discovered the course, can I still join?"
search_results = search(index, question)

## Prompt engineering

In [18]:
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")

    return "\n".join(lines).strip()

In [19]:
context = build_context(search_results)

In [20]:
print(context)

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: Certificate: Can I follow the course in a self-paced mode and get a certificate?
A: No, you can only get a certificate if you finish the course with a "live" cohort.

We don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.

You can only peer-review projects at the time the course is running; after the form is closed and the peer-review list is compiled.

General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you 

In [21]:
USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""

In [22]:
def build_prompt(question, search_results):
	context = build_context(search_results)
	prompt = USER_PROMPT_TEMPLATE.format(
		question=question,
		context=context
  )

	return prompt.strip()

In [23]:
prompt = build_prompt(question, search_results)

In [24]:
print(prompt)

Question:
i just discovered the course, can I still join?

Context:
General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: Certificate: Can I follow the course in a self-paced mode and get a certificate?
A: No, you can only get a certificate if you finish the course with a "live" cohort.

We don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.

You can only peer-review projects at the time the course is running; after the form is closed and the peer-review list is compiled.

General Course-Related Questions
Q: I just discovered the course. Can 

## RAG pipeline

In [25]:
INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""

In [38]:
def rag(instructions, user_prompt, index, model="gpt-4o-mini"):
	search_results = search(index, question)
	prompt = build_prompt(user_prompt, search_results)

	message_history = [
		{ 'role': 'system', 'content': instructions },
		{ 'role': 'user', 'content': prompt }
	]

	response = get_response(openai_client, prompt=message_history)

	return response

In [39]:
response = rag(INSTRUCTIONS, question, index=index)

In [40]:
response.choices[0].message.content

'Yes, you can still join the course. However, if you want to receive a certificate, you need to submit your project while the course is still accepting submissions.'

In [31]:
question

'i just discovered the course, can I still join?'

In [ ]:
INS